# POS and WordNet Sense Census

**Primary author:** Victoria

**Builds on:**
- *01_wn_filtering_and_split.ipynb* (Victoria — WordNet filtering and vocabulary construction)
- *03_train_g1.ipynb* (Victoria/Nathan — triplet construction)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Descriptive census of part-of-speech distributions and WordNet sense
properties across the vocabulary, the g1 training triplets, and the
validation pairs. The notebook characterises the scale of design issues
DI-2 (POS-biased sense selection) and DI-3 (unreliable frequency
ordering) for Stage 6 hypothesis testing, and establishes the POS
composition of both training and evaluation data. It also looks ahead
to the design of future phrase construction strategies by quantifying
how much sense evidence WordNet actually offers per word.

Reads: `data/filtered_split/wn_synset/vocabulary.csv`, `data/filtered_split/wn_synset/clues_wn_filtered.csv`, `data/filtered_split/wn_synset/clues_val.csv`, `data/triplets/g1_train.csv`, and WordNet via NLTK.
Produces: `outputs/pos_wordnet_census-results.md`, `outputs/pos_mismatch_examples.md`, and three PNG figures under `outputs/figures/`.

## §1 — Setup and data loading

Standard imports, environment auto-detection, and paths pinned via
`pathlib`. The notebook lives at
`custom_embedding_model/planning/exploration/`, so `DATA_DIR` and
`OUTPUT_DIR` are resolved relative to that location.

In [ ]:
# === Setup: imports, paths, environment
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import nltk
from nltk.corpus import wordnet as wn

try:
    wn.synsets('test')
except LookupError:
    nltk.download('wordnet', quiet=True)

# --- Environment auto-detection ---
try:
    IS_COLAB = 'google.colab' in str(get_ipython())
except NameError:
    IS_COLAB = False
IS_GREATLAKES = Path('/nfs/turbo').exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
    COMPONENT_ROOT = PROJECT_ROOT / 'custom_embedding_model'
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / 'ccc-project'
    COMPONENT_ROOT = PROJECT_ROOT / 'custom_embedding_model'
else:
    # Local: notebook lives at custom_embedding_model/planning/exploration/
    COMPONENT_ROOT = Path('../..').resolve()

DATA_DIR   = COMPONENT_ROOT / 'data'
OUTPUT_DIR = COMPONENT_ROOT / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_PATH     = DATA_DIR / 'filtered_split' / 'wn_synset' / 'vocabulary.csv'
CLUES_WN_PATH  = DATA_DIR / 'filtered_split' / 'wn_synset' / 'clues_wn_filtered.csv'
CLUES_VAL_PATH = DATA_DIR / 'filtered_split' / 'wn_synset' / 'clues_val.csv'
TRIPLETS_PATH  = DATA_DIR / 'triplets' / 'g1_train.csv'

RESULTS_PATH  = OUTPUT_DIR / 'pos_wordnet_census-results.md'
MISMATCH_PATH = OUTPUT_DIR / 'pos_mismatch_examples.md'
FIG_VOCAB     = FIGURE_DIR / 'pos_vocab_distribution.png'
FIG_TRIPLET   = FIGURE_DIR / 'pos_triplet_composition.png'
FIG_VAL       = FIGURE_DIR / 'pos_validation_composition.png'

for p in [VOCAB_PATH, CLUES_WN_PATH, CLUES_VAL_PATH, TRIPLETS_PATH]:
    assert p.exists(), f'Missing input: {p}'

print(f'Environment:    {"Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")}')
print(f'COMPONENT_ROOT: {COMPONENT_ROOT}')
print(f'FIGURE_DIR:     {FIGURE_DIR}')

In [ ]:
# === Version reporting (Decision 18)
import matplotlib
import spacy

print(f'pandas:     {pd.__version__}')
print(f'numpy:      {np.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn:    {sns.__version__}')
print(f'nltk:       {nltk.__version__}')
print(f'WordNet:    {wn.get_version()}')
print(f'spacy:      {spacy.__version__}')

### Load all four inputs

All CSVs are loaded with `keep_default_na=False` and `na_values=['']` so
that strings like `nan` (the crossword word for grandmother) survive.
Row-count assertions guard against silent version drift in the upstream
pipeline.

In [ ]:
# === Load inputs
vocab = pd.read_csv(VOCAB_PATH, keep_default_na=False, na_values=[''])
assert len(vocab) == 53_930, f'vocabulary rows: {len(vocab):,}'
print(f'vocabulary.csv:        {len(vocab):,} rows')

clues_wn = pd.read_csv(CLUES_WN_PATH, keep_default_na=False, na_values=[''])
clues_train_wn = clues_wn[clues_wn['split'] == 'train'].copy()
print(f'clues_wn_filtered.csv: {len(clues_wn):,} rows  '
      f'(train subset: {len(clues_train_wn):,})')

clues_val = pd.read_csv(CLUES_VAL_PATH, keep_default_na=False, na_values=[''])
assert len(clues_val) == 47_933, f'val rows: {len(clues_val):,}'
print(f'clues_val.csv:         {len(clues_val):,} rows')

triplets = pd.read_csv(TRIPLETS_PATH, keep_default_na=False, na_values=[''])
assert len(triplets) == 69_921, f'triplet rows: {len(triplets):,}'
print(f'g1_train.csv:          {len(triplets):,} rows')

## §2 — WordNet census: full vocabulary

For every word in the wn_synset vocabulary, we record synset counts by
POS, the POS and lemma-count statistics of `synsets(word)[0]`, and
reliability flags that tell us whether WordNet's frequency ordering is
informative at all. WordNet distinguishes the `a` (adjective) and `s`
(satellite adjective) POS letters internally; most of the reporting
collapses them into a combined `adj` category, while `n_synsets_a` and
`n_synsets_s` remain separate columns for completeness.

In [ ]:
# === Vocabulary census computation
# For each vocab word, summarise synset counts by POS, the POS and lemma
# count of sense[0], and reliability flags that tell us whether WordNet's
# frequency ordering is actually informative.

def _combine_adj(p):
    return 'a' if p in ('a', 's') else p

def census_word(word):
    syns = wn.synsets(word)
    if not syns:
        return None
    s0 = syns[0]
    s0_pos = s0.pos()

    def lemma_count(syn, w):
        # Sum counts across lemmas whose surface form equals the word.
        return sum(l.count() for l in syn.lemmas() if l.name() == w)

    s0_count = lemma_count(s0, word)
    by_pos = {'n': 0, 'v': 0, 'a': 0, 's': 0, 'r': 0}
    max_any = 0
    max_same = 0
    any_nonzero = False
    higher_other = False

    for syn in syns:
        by_pos[syn.pos()] += 1
        c = lemma_count(syn, word)
        if c > 0:
            any_nonzero = True
        if c > max_any:
            max_any = c
        if syn.pos() == s0_pos and c > max_same:
            max_same = c
        if syn.pos() != s0_pos and c > s0_count:
            higher_other = True

    return {
        'word': word,
        'n_synsets': len(syns),
        'n_synsets_n': by_pos['n'],
        'n_synsets_v': by_pos['v'],
        'n_synsets_a': by_pos['a'],
        'n_synsets_s': by_pos['s'],
        'n_synsets_r': by_pos['r'],
        'sense0_pos': s0_pos,
        'sense0_pos_combined': _combine_adj(s0_pos),
        'sense0_lemma_count': s0_count,
        'max_lemma_count_any_sense': max_any,
        'max_lemma_count_same_pos': max_same,
        'has_nonzero_count': any_nonzero,
        'sense0_is_max_within_pos': (s0_count == max_same),
        'higher_freq_other_pos': higher_other,
    }

t0 = time.time()
records = [census_word(w) for w in vocab['word']]
assert all(r is not None for r in records), 'vocab word with no synsets'
vocab_census = pd.DataFrame.from_records(records)
elapsed_census = time.time() - t0
print(f'Computed vocab census for {len(vocab_census):,} words in {elapsed_census:.1f}s')
vocab_census.head()

### §2a — POS distribution of sense[0]

Raw counts for all five WordNet POS labels are reported first, then the
combined `adj = a ∪ s` breakdown used in downstream comparisons.

In [ ]:
# === §2a: sense[0] POS distribution
POS_LABELS = {'n': 'noun', 'v': 'verb', 'a': 'adj', 's': 'sat_adj', 'r': 'adv'}
COMBINED_LABELS = {'n': 'noun', 'v': 'verb', 'a': 'adj (a+s)', 'r': 'adv'}

pos_raw = vocab_census['sense0_pos'].value_counts().reindex(
    ['n', 'v', 'a', 's', 'r'], fill_value=0
)
pos_combined = vocab_census['sense0_pos_combined'].value_counts().reindex(
    ['n', 'v', 'a', 'r'], fill_value=0
)
n_total = len(vocab_census)

print('sense[0] POS distribution (raw WN labels):')
for p, n in pos_raw.items():
    print(f'  {p} ({POS_LABELS[p]:8s}): {n:>6,}  ({n/n_total:.1%})')
print()
print('sense[0] POS distribution (a+s combined as "adj"):')
for p, n in pos_combined.items():
    print(f'  {p} ({COMBINED_LABELS[p]:10s}): {n:>6,}  ({n/n_total:.1%})')

### §2b — Sense availability

How much sense evidence does WordNet actually offer? Summary statistics on
`n_synsets`, the fraction of words that appear in multiple POS
categories, and a breakdown of words that live in exactly one POS.

In [ ]:
# === §2b: Sense availability
print('n_synsets per word (full vocabulary):')
print(vocab_census['n_synsets'].describe())
print()

# Per-POS presence, collapsing a+s into adj for this analysis
has = pd.DataFrame({
    'n': vocab_census['n_synsets_n'] > 0,
    'v': vocab_census['n_synsets_v'] > 0,
    'a': (vocab_census['n_synsets_a'] + vocab_census['n_synsets_s']) > 0,
    'r': vocab_census['n_synsets_r'] > 0,
})
n_pos = has.sum(axis=1)
print(f'Words with synsets in ≥ 2 POS categories: '
      f'{(n_pos >= 2).sum():,} ({(n_pos >= 2).mean():.1%})')
print()
print('Words with synsets in exactly one POS category:')
for p in ['n', 'v', 'a', 'r']:
    mask = (n_pos == 1) & has[p]
    print(f'  only {COMBINED_LABELS[p]:10s}: {mask.sum():>6,}  ({mask.mean():.1%})')

### §2c — Lemma count reliability

WordNet orders synsets for a word by how often a tagged corpus picked
them, but most vocabulary words have no tagged-corpus evidence at all —
so the `synsets(word)[0]` ordering is arbitrary for them. This cell
quantifies how often that happens and, among words that do have
evidence, how often `sense[0]` is *not* the most frequent sense of its
own POS (or how often a different POS has a strictly higher count).

In [ ]:
# === §2c: Lemma count reliability
hnz = vocab_census['has_nonzero_count']
print(f'Any nonzero lemma count:               {hnz.sum():>6,}  ({hnz.mean():.1%})')
print(f'All lemma counts are zero (arbitrary): {(~hnz).sum():>6,}  ({(~hnz).mean():.1%})')
print()

nonzero = vocab_census[hnz]
print(f'Among the {len(nonzero):,} words with nonzero counts:')
is_max = nonzero['sense0_is_max_within_pos']
ho     = nonzero['higher_freq_other_pos']
print(f'  sense[0] is max within its POS:        {is_max.sum():>6,}  ({is_max.mean():.1%})')
print(f'  a different POS has higher frequency:  {ho.sum():>6,}  ({ho.mean():.1%})')
print()

print('Crosstab: sense[0] POS (combined) × has_nonzero_count')
print(pd.crosstab(
    vocab_census['sense0_pos_combined'].map(COMBINED_LABELS),
    vocab_census['has_nonzero_count'],
    margins=True,
))

# --- Three-way reliability breakdown -------------------------------
# A word with exactly one synset has no sense selection to get wrong,
# so it is trivially correct for both DI-2 and DI-3 — it should not be
# lumped in with multi-synset words that have zero lemma counts.
is_unambig = vocab_census['n_synsets'] == 1
is_freq_conf = (
    (~is_unambig)
    & vocab_census['has_nonzero_count']
    & vocab_census['sense0_is_max_within_pos']
)
is_arbitrary = (~is_unambig) & (~is_freq_conf)
print()
print('Three-way reliability breakdown:')
print(f'  unambiguous (n_synsets == 1):       '
      f'{is_unambig.sum():>6,}  ({is_unambig.mean():.1%})')
print(f'  frequency-confirmed:                '
      f'{is_freq_conf.sum():>6,}  ({is_freq_conf.mean():.1%})')
print(f'  arbitrary (multi-synset, no proof): '
      f'{is_arbitrary.sum():>6,}  ({is_arbitrary.mean():.1%})')

### §2d — Multi-panel vocabulary figure

Three panels side by side: the combined sense[0] POS distribution, the
`n_synsets` histogram (clipped at 20 to keep the long tail legible), and
a reliability heatmap with three categories — **unambiguous**
(`n_synsets == 1`, no sense selection to get wrong), **frequency-
confirmed** (multi-synset and sense[0] is the max-frequency sense of its
POS), and **arbitrary** (multi-synset with no evidence or contrary
evidence for sense[0]).

In [ ]:
# === §2d: Multi-panel vocabulary figure
# Three-way reliability classification. Single-synset words are trivially
# correct for sense selection — they belong in their own bucket, not
# lumped with multi-synset words that lack frequency evidence.
vocab_census['reliability'] = np.where(
    vocab_census['n_synsets'] == 1,
    'unambiguous',
    np.where(
        vocab_census['has_nonzero_count']
        & vocab_census['sense0_is_max_within_pos'],
        'frequency-confirmed',
        'arbitrary',
    ),
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: sense[0] POS bars (combined)
order = ['n', 'v', 'a', 'r']
labels = ['noun', 'verb', 'adj (a+s)', 'adv']
counts_p1 = pos_combined.reindex(order, fill_value=0).values
bars = axes[0].bar(labels, counts_p1, edgecolor='black')
axes[0].set_title('sense[0] POS distribution\n(53,930 vocabulary words)')
axes[0].set_ylabel('Number of words')
for i, v in enumerate(counts_p1):
    axes[0].text(i, v + 400, f'{v:,}\n{v/n_total:.1%}', ha='center', fontsize=9)
axes[0].set_ylim(0, max(counts_p1) * 1.15)

# Panel 2: n_synsets histogram, clipped at 20
clipped = np.clip(vocab_census['n_synsets'].to_numpy(), 1, 20)
axes[1].hist(clipped, bins=np.arange(1, 22), edgecolor='black')
axes[1].set_xlabel('n_synsets per word (≥ 20 in last bin)')
axes[1].set_ylabel('Number of words')
axes[1].set_title('Sense availability per word')
axes[1].set_xticks(range(1, 21, 2))

# Panel 3: reliability heatmap (three categories)
RELIABILITY_ORDER = ['unambiguous', 'frequency-confirmed', 'arbitrary']
heat = pd.crosstab(
    vocab_census['sense0_pos_combined'],
    vocab_census['reliability'],
).reindex(index=order, columns=RELIABILITY_ORDER, fill_value=0)
heat.index = labels
sns.heatmap(heat, annot=True, fmt=',', cmap='YlOrRd', ax=axes[2],
            cbar_kws={'label': 'count'})
axes[2].set_title('sense[0] reliability (three-way)\n'
                  'unambiguous = 1 synset; freq-confirmed = max within POS;\n'
                  'arbitrary = multi-synset without evidence')
axes[2].set_xlabel('Reliability category')
axes[2].set_ylabel('sense[0] POS')

fig.tight_layout()
fig.savefig(FIG_VOCAB, dpi=300)
plt.show()
print(f'Saved: {FIG_VOCAB}')

## §3 — Contextual POS of definitions in clue surfaces

The contextual POS of a definition is the POS spaCy assigns to the
definition word(s) *as they appear inside the surface reading*. This is
what `g(f_clue(def))` is actually being asked to encode, so it is the
POS relevant to the f_clue anchor in training and the T=1 condition in
evaluation.

**Procedure** (per row):
1. Use `definition_wn` (articles already stripped per Decision 16); map
   underscores to spaces.
2. If `definition_wn` sits sentence-initially in the surface, lowercase
   the surface's first character before tagging — otherwise spaCy will
   often label a capitalised common noun as `PROPN`.
3. Tag with spaCy (`en_core_web_sm`) and locate the token span that
   matches `definition_wn`.
4. Single-word definition → take the POS of the matched token.
   Multi-word definition → take the POS of the last token, and accept
   only if `wn.synsets(definition_wn)` has exactly one synset whose POS
   agrees. Otherwise mark the row **undetermined** and collect it for
   review.

POS tagging is done once over unique `(surface, definition_wn)` pairs
drawn from both the validation split and the g1 training triplets, then
joined back to each row. Tagging runs through `nlp.pipe` in batches.

In [ ]:
# === spaCy setup and contextual POS helpers
# Only POS-tagging is needed — disable parser/ner/lemmatizer for speed.
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'lemmatizer'])

# spaCy UD tags → WordNet POS letter. Anything unmapped collapses to 'other'
# (most commonly DET / ADP / PART for clue connective words).
SPACY_TO_WN = {
    'NOUN': 'n', 'PROPN': 'n',
    'VERB': 'v', 'AUX':  'v',
    'ADJ':  'a',
    'ADV':  'r',
}

def to_wn_pos(spacy_pos):
    return SPACY_TO_WN.get(spacy_pos, 'other')

def prep_surface(surface, def_spaced):
    """Lowercase the first character of the surface when def_spaced appears
    sentence-initially. Sentence-initial capitals often push spaCy toward
    PROPN for what is really a common noun used as the definition.
    """
    if not surface:
        return surface
    if surface.lower().startswith(def_spaced.lower()):
        return surface[0].lower() + surface[1:]
    return surface

def find_def_span(doc, def_spaced):
    """Return (start, end) token indices matching def_spaced, or None.
    Case-insensitive match against contiguous token text."""
    target = def_spaced.lower().split()
    n = len(target)
    toks = [t.text.lower() for t in doc]
    for i in range(len(toks) - n + 1):
        if toks[i:i + n] == target:
            return (i, i + n)
    return None

def contextual_pos_for_pair(doc, def_spaced, def_wn):
    """Return (contextual_pos, spacy_pos_last, status).
    contextual_pos is a WN letter ('n'/'v'/'a'/'r'/'other') or None.
    status is 'determined' / 'undetermined' / 'span_not_found'.
    """
    span = find_def_span(doc, def_spaced)
    if span is None:
        return (None, None, 'span_not_found')
    start, end = span
    is_multiword = '_' in def_wn
    last_spacy_pos = doc[end - 1].pos_
    if not is_multiword:
        return (to_wn_pos(last_spacy_pos), last_spacy_pos, 'determined')
    last_wn = to_wn_pos(last_spacy_pos)
    syns = wn.synsets(def_wn)
    if len(syns) == 1 and _combine_adj(syns[0].pos()) == last_wn:
        return (last_wn, last_spacy_pos, 'determined')
    return (None, last_spacy_pos, 'undetermined')

In [ ]:
# === Build the set of (surface, definition_wn) pairs requiring tagging
val_subset = clues_val[['clue_id', 'definition', 'surface', 'definition_wn']].copy()
val_subset['source'] = 'val'

train_subset = triplets[['clue_id', 'definition']].merge(
    clues_train_wn[['clue_id', 'definition', 'surface', 'definition_wn']],
    on=['clue_id', 'definition'],
    how='left',
)
train_subset['source'] = 'train'

n_missing = train_subset['surface'].isna().sum()
assert n_missing == 0, (
    f'{n_missing:,} triplet rows failed to join clues_wn_filtered.csv on '
    '(clue_id, definition)'
)
print(f'Rows needing POS tagging: val={len(val_subset):,}, '
      f'train={len(train_subset):,}, total={len(val_subset) + len(train_subset):,}')

pairs = (
    pd.concat([val_subset, train_subset], ignore_index=True)
      [['surface', 'definition_wn']]
      .drop_duplicates()
      .reset_index(drop=True)
)
pairs['def_spaced'] = pairs['definition_wn'].str.replace('_', ' ')
pairs['prepped_surface'] = [
    prep_surface(s, d) for s, d in zip(pairs['surface'], pairs['def_spaced'])
]
print(f'Unique (surface, definition_wn) pairs to tag: {len(pairs):,}')

In [ ]:
# === POS-tag unique pairs with spaCy (batched)
t0 = time.time()
docs = list(nlp.pipe(pairs['prepped_surface'].tolist(), batch_size=1000))
elapsed_tag = time.time() - t0
print(f'spaCy tagged {len(docs):,} surfaces in {elapsed_tag:.1f}s '
      f'({elapsed_tag / len(docs) * 1000:.2f} ms/surface)')

In [ ]:
# === Assign contextual POS per (surface, definition_wn) pair
ctx_pos, spacy_last, status = [], [], []
for doc, def_spaced, def_wn in zip(docs, pairs['def_spaced'], pairs['definition_wn']):
    cp, sp, st = contextual_pos_for_pair(doc, def_spaced, def_wn)
    ctx_pos.append(cp)
    spacy_last.append(sp)
    status.append(st)
pairs['contextual_pos'] = ctx_pos
pairs['spacy_last_pos'] = spacy_last
pairs['context_status'] = status

print('Unique-pair tagging status:')
print(pairs['context_status'].value_counts())

# Merge POS assignments back onto the per-row val and train subsets
pos_map = pairs.set_index(['surface', 'definition_wn'])[
    ['contextual_pos', 'spacy_last_pos', 'context_status']
]
val_subset = val_subset.merge(
    pos_map, left_on=['surface', 'definition_wn'], right_index=True, how='left'
)
train_subset = train_subset.merge(
    pos_map, left_on=['surface', 'definition_wn'], right_index=True, how='left'
)

print()
print('Row-level coverage:')
for name, sub in [('val', val_subset), ('train', train_subset)]:
    counts = sub['context_status'].value_counts()
    n = len(sub)
    det = counts.get('determined', 0)
    und = counts.get('undetermined', 0)
    snf = counts.get('span_not_found', 0)
    print(f'  {name:5s} total={n:,}  determined={det:,} ({det/n:.1%})  '
          f'undetermined={und:,} ({und/n:.1%})  '
          f'span_not_found={snf:,} ({snf/n:.1%})')

### Save undetermined / unmatched examples

Rows where the contextual POS could not be confidently assigned are
written to `outputs/pos_mismatch_examples.md` so Victoria can skim them
without re-running the notebook.

In [ ]:
# === Save undetermined examples for review
bad = pairs[pairs['context_status'] != 'determined'].copy()
bad_sample = bad.head(60)

lines = [
    '# POS Mismatch / Undetermined Examples',
    '',
    'Unique `(surface, definition_wn)` pairs where contextual POS assignment',
    'failed. Either the definition span could not be located in the tagged',
    'surface (`span_not_found`) or the last-word POS heuristic did not',
    'agree with a unique WordNet POS (`undetermined`).',
    '',
    f'Total unique pairs marked: {len(bad):,}',
    '',
    '| status | definition_wn | spacy_last_pos | surface |',
    '| --- | --- | --- | --- |',
]
for _, r in bad_sample.iterrows():
    sp = r['spacy_last_pos'] if r['spacy_last_pos'] else '-'
    surface = str(r['surface']).replace('|', '\\|')
    lines.append(
        f"| {r['context_status']} | `{r['definition_wn']}` | {sp} | {surface} |"
    )
MISMATCH_PATH.write_text('\n'.join(lines) + '\n')
print(f'Saved {len(bad):,} undetermined pairs to {MISMATCH_PATH} '
      f'(showing first {len(bad_sample)})')

## §4 — POS of training triplets

Classify POS for all three triplet roles: anchor (contextual POS of the
f_clue definition *and* WordNet sense[0] POS of `definition_wn`),
positive (`answer_wn`), and negative (`distractor_wn`). Roles that go
through `f_common_wndef` use the WordNet sense[0] POS recorded in the
vocab census; the anchor additionally gets the contextual POS from §3
for comparison.

In [ ]:
# === Assemble the triplet-POS dataframe
vc_idx = vocab_census.set_index('word')

trip = train_subset.merge(
    triplets[['clue_id', 'definition', 'answer_wn', 'distractor_wn']],
    on=['clue_id', 'definition'],
    how='left',
)

# Pull vocab-census columns onto each role via .map. Words outside the
# census yield NaN; we report the (small) miss rate below.
VC_COLS = [
    'sense0_pos_combined',
    'has_nonzero_count',
    'sense0_is_max_within_pos',
    'higher_freq_other_pos',
    'n_synsets',
]

for role, prefix in [
    ('definition_wn', 'anchor_wn'),
    ('answer_wn',     'pos_wn'),
    ('distractor_wn', 'neg_wn'),
]:
    for col in VC_COLS:
        trip[f'{prefix}_{col}'] = trip[role].map(vc_idx[col])

# Sanity: every triplet word should live in the wn_synset vocabulary.
for role in ['definition_wn', 'answer_wn', 'distractor_wn']:
    miss = trip[role].map(lambda w: w not in vc_idx.index).sum()
    print(f'  {role} rows not in vocab_census: {miss:,}')
print(f'Triplet-POS dataframe: {len(trip):,} rows')
trip[['clue_id', 'definition_wn', 'contextual_pos', 'anchor_wn_sense0_pos_combined',
      'answer_wn', 'pos_wn_sense0_pos_combined',
      'distractor_wn', 'neg_wn_sense0_pos_combined']].head()

### §4a — Per-role POS distribution

Percentage noun vs. non-noun for every role. The anchor is reported
twice: once using the contextual POS (what f_clue actually anchors on)
and once using `definition_wn`'s WordNet sense[0] POS (what the model
would see if the definition were used decontextualised).

In [ ]:
# === §4a: Per-role POS distribution
def pos_breakdown(series, label, include_undet=False):
    n = len(series)
    s = series.fillna('undetermined') if include_undet else series
    counts = s.value_counts()
    print(f'{label}  (n={n:,})')
    order = ['n', 'v', 'a', 'r', 'other', 'undetermined']
    for key in order:
        if key in counts.index:
            c = counts[key]
            print(f'  {key:12s}: {c:>7,}  ({c/n:.1%})')
    print()

pos_breakdown(trip['contextual_pos'], 'Anchor — contextual POS', include_undet=True)
pos_breakdown(trip['anchor_wn_sense0_pos_combined'], 'Anchor — WN sense[0] POS of definition_wn')
pos_breakdown(trip['pos_wn_sense0_pos_combined'], 'Positive (answer_wn) — WN sense[0] POS')
pos_breakdown(trip['neg_wn_sense0_pos_combined'], 'Negative (distractor_wn) — WN sense[0] POS')

# Reliability summary for pos/neg roles
for role, label in [('pos_wn', 'Positive'), ('neg_wn', 'Negative')]:
    hnz = trip[f'{role}_has_nonzero_count'].sum()
    ism = trip[f'{role}_sense0_is_max_within_pos'].sum()
    print(f'{label}: has_nonzero_count = {hnz:,} ({hnz/len(trip):.1%}); '
          f'sense[0] is max within POS = {ism:,} ({ism/len(trip):.1%})')

### §4b — POS mismatch: anchor contextual vs. WordNet sense[0]

Where does f_clue put the model in a different POS lane from what the
sense[0] convention would pick? The crosstab shows disagreements with
their direction.

In [ ]:
# === §4b: Contextual POS vs. WN sense[0] POS (anchor)
det = trip[trip['context_status'] == 'determined'].copy()
trip_agree = (det['contextual_pos'] == det['anchor_wn_sense0_pos_combined']).sum()
trip_n_det = len(det)
print(f'Rows with a determined contextual POS: {trip_n_det:,} '
      f'({trip_n_det/len(trip):.1%} of triplets)')
print(f'  contextual == WN sense[0]: {trip_agree:,} ({trip_agree/trip_n_det:.1%})')
print(f'  disagreement:              {trip_n_det-trip_agree:,} '
      f'({(trip_n_det-trip_agree)/trip_n_det:.1%})')
print()

ct = pd.crosstab(
    det['contextual_pos'].rename('contextual'),
    det['anchor_wn_sense0_pos_combined'].rename('wn_sense0'),
).reindex(index=['n', 'v', 'a', 'r', 'other'], columns=['n', 'v', 'a', 'r'], fill_value=0)
print('Crosstab: anchor contextual POS (rows) × anchor WN sense[0] POS (cols)')
print(ct)
trip_anchor_crosstab = ct

### §4c — Triplet-level composition

Composition across roles: how often is the full wndef side of a triplet
(positive + negative) purely noun-noun? And when positive and negative
are both nouns, does the anchor's contextual POS follow, or does f_clue
tag the anchor as something non-nominal?

In [ ]:
# === §4c: Triplet-level composition
is_pos_n = trip['pos_wn_sense0_pos_combined'] == 'n'
is_neg_n = trip['neg_wn_sense0_pos_combined'] == 'n'
both_n = is_pos_n & is_neg_n

print(f'Triplets where both positive AND negative are noun: '
      f'{both_n.sum():,} ({both_n.mean():.1%})')
print(f'Triplets where positive is noun:                     '
      f'{is_pos_n.sum():,} ({is_pos_n.mean():.1%})')
print(f'Triplets where negative is noun:                     '
      f'{is_neg_n.sum():,} ({is_neg_n.mean():.1%})')
print()

print('Anchor contextual POS distribution, conditional on positive+negative POS:')
print('  When BOTH pos and neg are noun:')
b = trip[both_n]
for key, c in b['contextual_pos'].fillna('undetermined').value_counts().items():
    print(f'    {str(key):12s}: {c:>6,}  ({c/len(b):.1%})')
print('  When at least one of pos/neg is NOT noun:')
nb = trip[~both_n]
for key, c in nb['contextual_pos'].fillna('undetermined').value_counts().items():
    print(f'    {str(key):12s}: {c:>6,}  ({c/len(nb):.1%})')

### §4d — Sense reliability in training

How often is the anchor or a wndef role trained on an arbitrary sense
ordering (zero lemma counts anywhere) versus a reliably ordered one?
And how often does the training data push the model toward a noun sense
when a more frequent non-noun sense exists?

The existing `trip_both_hnz` / `trip_any_arb` statistics treat every
word with zero lemma counts as "arbitrary" — but a single-synset word
has no sense selection to get wrong. The supplementary three-way block
below reclassifies single-synset words as **unambiguous** and only
counts multi-synset words without frequency evidence as genuinely
arbitrary. Both summaries are reported.

In [ ]:
# === §4d: Sense reliability across triplets
trip_both_hnz = trip['pos_wn_has_nonzero_count'] & trip['neg_wn_has_nonzero_count']
trip_any_arb  = (~trip['pos_wn_has_nonzero_count']) | (~trip['neg_wn_has_nonzero_count'])
trip_any_higher_other = (
    trip['anchor_wn_higher_freq_other_pos'].fillna(False)
    | trip['pos_wn_higher_freq_other_pos'].fillna(False)
    | trip['neg_wn_higher_freq_other_pos'].fillna(False)
)
print(f'Both pos and neg have nonzero counts:              '
      f'{trip_both_hnz.sum():,} ({trip_both_hnz.mean():.1%})')
print(f'At least one role has all-zero counts (arbitrary): '
      f'{trip_any_arb.sum():,} ({trip_any_arb.mean():.1%})')
print(f'At least one role has higher-freq sense in other POS: '
      f'{trip_any_higher_other.sum():,} ({trip_any_higher_other.mean():.1%})')

# --- Three-way reliability (treating single-synset words as trivial) --
def role_reliability(df, prefix):
    n = df[f'{prefix}_n_synsets']
    hnz = df[f'{prefix}_has_nonzero_count']
    ism = df[f'{prefix}_sense0_is_max_within_pos']
    return np.where(
        n == 1, 'unambiguous',
        np.where(hnz & ism, 'frequency-confirmed', 'arbitrary'),
    )

trip['anchor_reliability'] = role_reliability(trip, 'anchor_wn')
trip['pos_reliability']    = role_reliability(trip, 'pos_wn')
trip['neg_reliability']    = role_reliability(trip, 'neg_wn')

trip_all_non_arb = (
    (trip['anchor_reliability'] != 'arbitrary')
    & (trip['pos_reliability']    != 'arbitrary')
    & (trip['neg_reliability']    != 'arbitrary')
)
print()
print('Three-way reliability (anchor + pos + neg, across 69,921 triplets):')
print(f'  All three roles unambiguous or freq-confirmed: '
      f'{trip_all_non_arb.sum():,} ({trip_all_non_arb.mean():.1%})')
print(f'  At least one role arbitrary:                   '
      f'{(~trip_all_non_arb).sum():,} ({(~trip_all_non_arb).mean():.1%})')
print()
print('Per-role reliability (training triplets):')
for role, label in [('anchor_reliability', 'anchor (definition_wn)'),
                    ('pos_reliability',    'positive (answer_wn)'),
                    ('neg_reliability',    'negative (distractor_wn)')]:
    vc = trip[role].value_counts().reindex(
        ['unambiguous', 'frequency-confirmed', 'arbitrary'], fill_value=0
    )
    print(f'  {label}:')
    for cat, c in vc.items():
        print(f'    {cat:20s}: {c:>6,}  ({c/len(trip):.1%})')

### §4e — Training triplet figure

Two panels: grouped bars of POS distribution by role, and a heatmap of
anchor contextual POS vs. anchor WordNet sense[0] POS (the diagonal is
agreement).

In [ ]:
# === §4e: Training triplet figure
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1: POS by role
order_x = ['n', 'v', 'a', 'r', 'other', 'undetermined']
role_cols = [
    ('Anchor contextual',      trip['contextual_pos'].fillna('undetermined')),
    ('Anchor WN sense[0]',     trip['anchor_wn_sense0_pos_combined']),
    ('Positive (answer_wn)',   trip['pos_wn_sense0_pos_combined']),
    ('Negative (distractor)',  trip['neg_wn_sense0_pos_combined']),
]
role_labels = [r[0] for r in role_cols]
n_roles = len(role_cols)
x = np.arange(len(order_x))
width = 0.2
for i, (name, s) in enumerate(role_cols):
    counts = s.value_counts().reindex(order_x, fill_value=0)
    frac = counts / len(trip)
    axes[0].bar(x + (i - (n_roles - 1) / 2) * width, frac.values, width,
                label=name, edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(order_x)
axes[0].set_ylabel('Fraction of triplets')
axes[0].set_title('POS distribution by triplet role  (g1_train, n=69,921)')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.0)

# Panel 2: anchor contextual vs WN sense[0] heatmap
sns.heatmap(trip_anchor_crosstab, annot=True, fmt=',', cmap='YlOrRd',
            ax=axes[1], cbar_kws={'label': 'count'})
axes[1].set_title('Anchor contextual POS (rows) vs. WN sense[0] POS (cols)\n'
                  'diagonal = agreement; off-diagonal = f_clue vs. wndef POS mismatch')
axes[1].set_xlabel('anchor_wn sense[0] POS')
axes[1].set_ylabel('contextual POS')

fig.tight_layout()
fig.savefig(FIG_TRIPLET, dpi=300)
plt.show()
print(f'Saved: {FIG_TRIPLET}')

## §5 — POS of validation pairs

For each validation row we care about three POS slots: the
clue-contextualised definition used in T=1 (contextual POS from §3),
the decontextualised definition used in T=0 (`definition_wn`'s
sense[0] POS), and the answer (`answer_wn`'s sense[0] POS). The
crosstabs below show whether the evaluation is dominated by
noun-noun comparisons and how often f_clue and `f_common_wndef` tag
the definition differently.

In [ ]:
# === Assemble the validation-pair POS dataframe
val = val_subset.copy()  # already has contextual_pos / context_status / definition_wn
val = val.merge(
    clues_val[['clue_id', 'definition', 'answer_wn']],
    on=['clue_id', 'definition'],
    how='left',
)

for role, prefix in [('definition_wn', 'def_wn'), ('answer_wn', 'ans_wn')]:
    for col in VC_COLS:
        val[f'{prefix}_{col}'] = val[role].map(vc_idx[col])

print(f'Validation pair dataframe: {len(val):,} rows')
val[['clue_id', 'definition_wn', 'contextual_pos', 'def_wn_sense0_pos_combined',
     'answer_wn', 'ans_wn_sense0_pos_combined']].head()

### §5a — Per-component POS distribution

Contextualised definition from spaCy (T=1 side), decontextualised
definition's WordNet sense[0] POS (T=0 side), and answer's sense[0] POS.

In [ ]:
# === §5a: Per-component POS distribution
pos_breakdown(val['contextual_pos'], 'Contextualised definition (T=1, spaCy)', include_undet=True)
pos_breakdown(val['def_wn_sense0_pos_combined'], 'Decontextualised definition (T=0, WN sense[0])')
pos_breakdown(val['ans_wn_sense0_pos_combined'], 'Answer (WN sense[0])')

for role, label in [('def_wn', 'Def (decontextualised)'), ('ans_wn', 'Answer')]:
    hnz = val[f'{role}_has_nonzero_count'].sum()
    print(f'{label}: has_nonzero_count = {hnz:,} ({hnz/len(val):.1%})')

### §5b — Pair-level composition for T=0 (definition × answer)

Both slots use `f_common_wndef`, so the relevant POS is WordNet
sense[0] POS on each side. The condensed 2×2 view (noun / not-noun)
quantifies how noun-noun the evaluation is.

In [ ]:
# === §5b: Pair-level composition for T=0
ct_full = pd.crosstab(
    val['def_wn_sense0_pos_combined'].rename('def_wn'),
    val['ans_wn_sense0_pos_combined'].rename('ans_wn'),
).reindex(index=['n', 'v', 'a', 'r'], columns=['n', 'v', 'a', 'r'], fill_value=0)
print('Full crosstab: definition sense[0] POS × answer sense[0] POS')
print(ct_full)
print()

# Condensed 2x2 (noun / not-noun)
def_noun = (val['def_wn_sense0_pos_combined'] == 'n')
ans_noun = (val['ans_wn_sense0_pos_combined'] == 'n')
condensed = pd.DataFrame({
    'ans_noun':     [(def_noun & ans_noun).sum(),  (~def_noun & ans_noun).sum()],
    'ans_not_noun': [(def_noun & ~ans_noun).sum(), (~def_noun & ~ans_noun).sum()],
}, index=['def_noun', 'def_not_noun'])
print('Condensed 2x2 (count):')
print(condensed)
print()
print('Condensed 2x2 (percentage of pairs):')
print((condensed / len(val) * 100).round(1).astype(str) + '%')
val_t0_condensed = condensed

### §5c — POS mismatch for T=1 (contextual def vs. answer sense[0])

When the definition comes in through f_clue, its contextual POS is what
the model sees. How often does that contextual POS agree with the
answer's WordNet sense[0] POS?

In [ ]:
# === §5c: POS mismatch for T=1
det_val = val[val['context_status'] == 'determined'].copy()
val_agree = (det_val['contextual_pos'] == det_val['ans_wn_sense0_pos_combined']).sum()
val_n_det = len(det_val)
print(f'Val rows with determined contextual POS: {val_n_det:,} '
      f'({val_n_det/len(val):.1%} of validation)')
print(f'  contextual def POS == answer sense[0] POS: '
      f'{val_agree:,} ({val_agree/val_n_det:.1%})')
print()

ct_t1 = pd.crosstab(
    det_val['contextual_pos'].rename('contextual_def'),
    det_val['ans_wn_sense0_pos_combined'].rename('ans_sense0'),
).reindex(index=['n', 'v', 'a', 'r', 'other'], columns=['n', 'v', 'a', 'r'], fill_value=0)
print('Crosstab: contextual def POS (rows) × answer WN sense[0] POS (cols)')
print(ct_t1)
val_t1_crosstab = ct_t1

### §5d — Sense reliability in evaluation

How many validation pairs are comparing two reliably ordered words vs.
at least one word whose synset ordering is arbitrary? As in §4d, we
report both the original `has_nonzero_count` statistic and the three-way
classification that correctly treats single-synset words as trivially
reliable.

In [ ]:
# === §5d: Sense reliability in evaluation
val_both_hnz = val['def_wn_has_nonzero_count'] & val['ans_wn_has_nonzero_count']
val_any_arb  = (~val['def_wn_has_nonzero_count']) | (~val['ans_wn_has_nonzero_count'])
print(f'Pairs where both def and ans have nonzero counts: '
      f'{val_both_hnz.sum():,} ({val_both_hnz.mean():.1%})')
print(f'Pairs with at least one arbitrary-ordering word:   '
      f'{val_any_arb.sum():,} ({val_any_arb.mean():.1%})')

# --- Three-way reliability (treating single-synset words as trivial) --
val['def_reliability'] = role_reliability(val, 'def_wn')
val['ans_reliability'] = role_reliability(val, 'ans_wn')

val_both_non_arb = (
    (val['def_reliability'] != 'arbitrary')
    & (val['ans_reliability'] != 'arbitrary')
)
print()
print('Three-way reliability (def + ans, across validation pairs):')
print(f'  Both roles unambiguous or freq-confirmed: '
      f'{val_both_non_arb.sum():,} ({val_both_non_arb.mean():.1%})')
print(f'  At least one role arbitrary:              '
      f'{(~val_both_non_arb).sum():,} ({(~val_both_non_arb).mean():.1%})')
print()
print('Per-role reliability (validation pairs):')
for role, label in [('def_reliability', 'definition_wn'),
                    ('ans_reliability', 'answer_wn')]:
    vc = val[role].value_counts().reindex(
        ['unambiguous', 'frequency-confirmed', 'arbitrary'], fill_value=0
    )
    print(f'  {label}:')
    for cat, c in vc.items():
        print(f'    {cat:20s}: {c:>6,}  ({c/len(val):.1%})')

### §5e — Validation pair figure

Two panels: an annotated 2×2 heatmap of the T=0 noun-noun dominance
pattern, and a grouped bar chart of POS distribution across the three
evaluation components.

In [ ]:
# === §5e: Validation pair figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: 2x2 noun/non-noun heatmap with counts + percentages
annot = val_t0_condensed.copy().astype(object)
for r in annot.index:
    for c in annot.columns:
        n = int(val_t0_condensed.loc[r, c])
        annot.loc[r, c] = f'{n:,}\n({n/len(val):.1%})'
sns.heatmap(val_t0_condensed, annot=annot.values, fmt='', cmap='YlOrRd',
            ax=axes[0], cbar_kws={'label': 'count'})
axes[0].set_title(f'T=0 pair composition (n={len(val):,})')
axes[0].set_xlabel('answer sense[0] POS')
axes[0].set_ylabel('definition sense[0] POS')

# Panel 2: POS distribution by evaluation component
comp_cols = [
    ('Contextual def (T=1)',  val['contextual_pos'].fillna('undetermined')),
    ('WN sense[0] def (T=0)', val['def_wn_sense0_pos_combined']),
    ('WN sense[0] answer',    val['ans_wn_sense0_pos_combined']),
]
n_comp = len(comp_cols)
x = np.arange(len(order_x))
width = 0.27
for i, (name, s) in enumerate(comp_cols):
    counts = s.value_counts().reindex(order_x, fill_value=0)
    frac = counts / len(val)
    axes[1].bar(x + (i - (n_comp - 1) / 2) * width, frac.values, width,
                label=name, edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(order_x)
axes[1].set_ylabel('Fraction of pairs')
axes[1].set_title(f'POS distribution by evaluation component  (n={len(val):,})')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.0)

fig.tight_layout()
fig.savefig(FIG_VAL, dpi=300)
plt.show()
print(f'Saved: {FIG_VAL}')

## §7 — Write results file

`outputs/pos_wordnet_census-results.md` captures all the numbers and
tables produced above, stamped with the exact package versions used
(Decision 18).

In [ ]:
# === Write results file
import datetime as _dt

# Three-way reliability counts for the results file (vocab census)
rel_counts = pd.crosstab(
    vocab_census['sense0_pos_combined'].map(COMBINED_LABELS),
    vocab_census['reliability'],
).reindex(index=[COMBINED_LABELS[p] for p in ['n','v','a','r']],
          columns=RELIABILITY_ORDER, fill_value=0)

def df_to_md(df, index_label='', float_fmt='{:,.0f}'):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join([index_label] + [str(c) for c in cols]) + ' |']
    lines.append('| ' + ' | '.join(['---'] * (len(cols) + 1)) + ' |')
    for idx, row in df.iterrows():
        vals = [float_fmt.format(v) if isinstance(v, (int, float, np.integer, np.floating)) else str(v) for v in row]
        lines.append('| ' + ' | '.join([str(idx)] + vals) + ' |')
    return '\n'.join(lines)

now = _dt.datetime.now().strftime('%Y-%m-%d')

sections = []
sections.append(f'# POS and WordNet Sense Census — Results\n')
sections.append(f'Generated: {now}  ')
sections.append('Notebook: `planning/exploration/pos_wordnet_census.ipynb`  ')
sections.append(f'Vocabulary rows: {len(vocab_census):,}  ')
sections.append(f'Validation rows: {len(val):,}  ')
sections.append(f'Training triplet rows: {len(trip):,}  ')
sections.append('')

sections.append('## Versions')
sections.append('')
sections.append(f'- pandas: {pd.__version__}')
sections.append(f'- numpy: {np.__version__}')
sections.append(f'- matplotlib: {matplotlib.__version__}')
sections.append(f'- seaborn: {sns.__version__}')
sections.append(f'- nltk: {nltk.__version__} (WordNet {wn.get_version()})')
sections.append(f'- spacy: {spacy.__version__} (en_core_web_sm, POS tagger)')
sections.append('')

# §2
sections.append('## §2 — Vocabulary census')
sections.append('')
sections.append('### sense[0] POS distribution (raw WN labels)')
raw_tbl = pd.DataFrame({
    'count': pos_raw.values,
    'fraction': (pos_raw.values / n_total),
}, index=[f'{p} ({POS_LABELS[p]})' for p in pos_raw.index])
raw_tbl['fraction'] = raw_tbl['fraction'].apply(lambda x: f'{x:.1%}')
sections.append(df_to_md(raw_tbl, index_label='pos'))
sections.append('')
sections.append('### sense[0] POS distribution (combined adj)')
comb_tbl = pd.DataFrame({
    'count': pos_combined.values,
    'fraction': [f'{v/n_total:.1%}' for v in pos_combined.values],
}, index=[COMBINED_LABELS[p] for p in pos_combined.index])
sections.append(df_to_md(comb_tbl, index_label='pos'))
sections.append('')
sections.append('### Sense availability')
desc = vocab_census['n_synsets'].describe()
sections.append(f'- mean={desc["mean"]:.2f}, median={int(desc["50%"])}, '
                f'q1={int(desc["25%"])}, q3={int(desc["75%"])}, '
                f'max={int(desc["max"])}')
sections.append(f'- Words with synsets in ≥ 2 POS categories: '
                f'{(n_pos >= 2).sum():,} ({(n_pos >= 2).mean():.1%})')
for p in ['n', 'v', 'a', 'r']:
    mask = (n_pos == 1) & has[p]
    sections.append(f'- Only {COMBINED_LABELS[p]}: {mask.sum():,} ({mask.mean():.1%})')
sections.append('')
sections.append('### Lemma count reliability')
sections.append(f'- Any nonzero lemma count: {vocab_census["has_nonzero_count"].sum():,} '
                f'({vocab_census["has_nonzero_count"].mean():.1%})')
sections.append(f'- Among those: sense[0] is max within POS = '
                f'{nonzero["sense0_is_max_within_pos"].sum():,} '
                f'({nonzero["sense0_is_max_within_pos"].mean():.1%})')
sections.append(f'- Among those: higher-freq sense in other POS = '
                f'{nonzero["higher_freq_other_pos"].sum():,} '
                f'({nonzero["higher_freq_other_pos"].mean():.1%})')
sections.append('')
sections.append('### Three-way reliability breakdown')
sections.append(f'- Unambiguous (n_synsets == 1):       '
                f'{is_unambig.sum():,} ({is_unambig.mean():.1%})')
sections.append(f'- Frequency-confirmed:                '
                f'{is_freq_conf.sum():,} ({is_freq_conf.mean():.1%})')
sections.append(f'- Arbitrary (multi-synset, no proof): '
                f'{is_arbitrary.sum():,} ({is_arbitrary.mean():.1%})')
sections.append('')
sections.append('### Reliability × sense[0] POS')
sections.append(df_to_md(rel_counts, index_label='pos'))
sections.append('')

# §3
sections.append('## §3 — Contextual POS coverage')
sections.append('')
for name, sub in [('validation', val_subset), ('training', train_subset)]:
    counts = sub['context_status'].value_counts()
    n = len(sub)
    det = counts.get('determined', 0)
    und = counts.get('undetermined', 0)
    snf = counts.get('span_not_found', 0)
    sections.append(f'- {name} (n={n:,}): determined={det:,} ({det/n:.1%}), '
                    f'undetermined={und:,} ({und/n:.1%}), '
                    f'span_not_found={snf:,} ({snf/n:.1%})')
sections.append(f'- Unique-pair undetermined examples saved to '
                f'`outputs/pos_mismatch_examples.md`')
sections.append('')

# §4
sections.append('## §4 — Training triplet POS')
sections.append('')
sections.append('### Per-role POS distribution (fraction of triplets)')
role_tbl = {}
for name, s in [
    ('anchor_contextual', trip['contextual_pos'].fillna('undetermined')),
    ('anchor_wn_sense0',  trip['anchor_wn_sense0_pos_combined']),
    ('positive_wn_sense0', trip['pos_wn_sense0_pos_combined']),
    ('negative_wn_sense0', trip['neg_wn_sense0_pos_combined']),
]:
    counts = s.value_counts().reindex(order_x, fill_value=0)
    role_tbl[name] = [f'{c/len(trip):.1%}' for c in counts.values]
role_df = pd.DataFrame(role_tbl, index=order_x)
sections.append(df_to_md(role_df, index_label='pos'))
sections.append('')
sections.append('### Anchor contextual vs. WN sense[0] crosstab')
sections.append(df_to_md(trip_anchor_crosstab, index_label='contextual↓ / wn→'))
sections.append(f'- Determined-row agreement: {trip_agree:,} / {trip_n_det:,} '
                f'({trip_agree/trip_n_det:.1%})')
sections.append('')
sections.append('### Triplet-level composition')
sections.append(f'- Both positive and negative are nouns: {both_n.sum():,} '
                f'({both_n.mean():.1%})')
sections.append(f'- Positive is noun: {is_pos_n.sum():,} ({is_pos_n.mean():.1%})')
sections.append(f'- Negative is noun: {is_neg_n.sum():,} ({is_neg_n.mean():.1%})')
sections.append('')
sections.append('### Sense reliability')
sections.append(f'- Both pos and neg have nonzero counts: {trip_both_hnz.sum():,} '
                f'({trip_both_hnz.mean():.1%})')
sections.append(f'- At least one role arbitrary: {trip_any_arb.sum():,} '
                f'({trip_any_arb.mean():.1%})')
sections.append(f'- At least one role has higher-freq other-POS sense: '
                f'{trip_any_higher_other.sum():,} ({trip_any_higher_other.mean():.1%})')
sections.append('')
sections.append('### Three-way reliability (triplet-level)')
sections.append(f'- All three roles unambiguous or frequency-confirmed: '
                f'{trip_all_non_arb.sum():,} ({trip_all_non_arb.mean():.1%})')
sections.append(f'- At least one role arbitrary: '
                f'{(~trip_all_non_arb).sum():,} ({(~trip_all_non_arb).mean():.1%})')
sections.append('')
sections.append('#### Per-role reliability (fraction of triplets)')
trip_role_tbl = {}
for col, name in [('anchor_reliability', 'anchor (definition_wn)'),
                  ('pos_reliability',    'positive (answer_wn)'),
                  ('neg_reliability',    'negative (distractor_wn)')]:
    vc = trip[col].value_counts().reindex(
        ['unambiguous', 'frequency-confirmed', 'arbitrary'], fill_value=0
    )
    trip_role_tbl[name] = [f'{c/len(trip):.1%}' for c in vc.values]
trip_role_df = pd.DataFrame(
    trip_role_tbl,
    index=['unambiguous', 'frequency-confirmed', 'arbitrary'],
)
sections.append(df_to_md(trip_role_df, index_label='category'))
sections.append('')

# §5
sections.append('## §5 — Validation pair POS')
sections.append('')
sections.append('### Per-component POS distribution (fraction of pairs)')
comp_tbl = {}
for name, s in [
    ('contextual_def_T1', val['contextual_pos'].fillna('undetermined')),
    ('wn_sense0_def_T0', val['def_wn_sense0_pos_combined']),
    ('wn_sense0_answer', val['ans_wn_sense0_pos_combined']),
]:
    counts = s.value_counts().reindex(order_x, fill_value=0)
    comp_tbl[name] = [f'{c/len(val):.1%}' for c in counts.values]
comp_df = pd.DataFrame(comp_tbl, index=order_x)
sections.append(df_to_md(comp_df, index_label='pos'))
sections.append('')
sections.append('### T=0 pair composition (def sense[0] POS × ans sense[0] POS)')
sections.append(df_to_md(ct_full, index_label='def↓ / ans→'))
sections.append('')
sections.append(f'- Condensed 2x2 noun-noun: {val_t0_condensed.loc["def_noun", "ans_noun"]:,} '
                f'({val_t0_condensed.loc["def_noun", "ans_noun"]/len(val):.1%}); '
                f'noun-other: {val_t0_condensed.loc["def_noun", "ans_not_noun"]:,} '
                f'({val_t0_condensed.loc["def_noun", "ans_not_noun"]/len(val):.1%}); '
                f'other-noun: {val_t0_condensed.loc["def_not_noun", "ans_noun"]:,} '
                f'({val_t0_condensed.loc["def_not_noun", "ans_noun"]/len(val):.1%}); '
                f'other-other: {val_t0_condensed.loc["def_not_noun", "ans_not_noun"]:,} '
                f'({val_t0_condensed.loc["def_not_noun", "ans_not_noun"]/len(val):.1%})')
sections.append('')
sections.append('### T=1 pair composition (contextual def POS × ans sense[0] POS)')
sections.append(df_to_md(val_t1_crosstab, index_label='ctx↓ / ans→'))
sections.append(f'- Determined-row agreement: {val_agree:,} / {val_n_det:,} '
                f'({val_agree/val_n_det:.1%})')
sections.append('')
sections.append('### Sense reliability')
sections.append(f'- Both def and ans have nonzero counts: {val_both_hnz.sum():,} '
                f'({val_both_hnz.mean():.1%})')
sections.append(f'- At least one arbitrary: {val_any_arb.sum():,} '
                f'({val_any_arb.mean():.1%})')
sections.append('')
sections.append('### Three-way reliability (pair-level)')
sections.append(f'- Both roles unambiguous or frequency-confirmed: '
                f'{val_both_non_arb.sum():,} ({val_both_non_arb.mean():.1%})')
sections.append(f'- At least one role arbitrary: '
                f'{(~val_both_non_arb).sum():,} ({(~val_both_non_arb).mean():.1%})')
sections.append('')
sections.append('#### Per-role reliability (fraction of pairs)')
val_role_tbl = {}
for col, name in [('def_reliability', 'definition_wn'),
                  ('ans_reliability', 'answer_wn')]:
    vc = val[col].value_counts().reindex(
        ['unambiguous', 'frequency-confirmed', 'arbitrary'], fill_value=0
    )
    val_role_tbl[name] = [f'{c/len(val):.1%}' for c in vc.values]
val_role_df = pd.DataFrame(
    val_role_tbl,
    index=['unambiguous', 'frequency-confirmed', 'arbitrary'],
)
sections.append(df_to_md(val_role_df, index_label='category'))
sections.append('')

# Runtimes
sections.append('## Runtimes')
sections.append('')
sections.append(f'- Vocab census: {elapsed_census:.1f}s ({len(vocab_census):,} words)')
sections.append(f'- spaCy POS tagging: {elapsed_tag:.1f}s ({len(docs):,} unique surfaces)')
sections.append('')

RESULTS_PATH.write_text('\n'.join(sections))
print(f'Wrote {RESULTS_PATH}')

## §6 — Summary

**Scope and runtime.** Computed a WordNet census over all
53,930 vocabulary words (∼2–3s) and POS-tagged the unique surfaces
backing 47,933 validation pairs and 69,921 g1 training triplets with
spaCy `en_core_web_sm` (see the cell above for the exact wall-clock
time). All three figures were written to `outputs/figures/`.

**Vocabulary findings (DI-2 and DI-3 scale).** The sense[0] POS of the
vocabulary is dominated by nouns, with a long polysemous tail. The
three-way reliability classification — **unambiguous** (one synset,
nothing to disambiguate), **frequency-confirmed** (multi-synset and
sense[0] is max within POS), and **arbitrary** (multi-synset without
evidence) — shows that the scale of DI-2 and DI-3 is smaller than a
naïve `has_nonzero_count` reading suggests: nearly half the vocabulary
has only one synset, so no sense selection is being made for those
words at all. Only the multi-synset-without-evidence category is the
genuinely problematic one. Exact numbers are recorded in
`outputs/pos_wordnet_census-results.md`.

**Training triplets.** The anchor's contextual POS often disagrees
with `definition_wn`'s WordNet sense[0] POS, which means f_clue and
`f_common_wndef` can place the anchor in different POS lanes.
Positives and negatives are overwhelmingly encoded via nominal senses,
so the triplet pushes the model toward noun-noun comparisons even when
the clue-contextualised definition is not nominal. The triplet-level
three-way reliability summary quantifies how many triplets have at
least one genuinely arbitrary role (multi-synset, no evidence) versus
triplets where every role is either unambiguous or frequency-confirmed.

**Validation pairs.** T=0 is dominated by noun-noun comparisons, and
T=1 inherits that asymmetry: the contextual POS of the definition
agrees with the answer's sense[0] POS less often than naïve POS
matching would predict. The pair-level three-way reliability summary
sets the scale for DI-3 in evaluation — many pairs involve at least
one single-synset word where no sense selection is happening at all.

**Forward-looking signal for future f's.** Roughly half the
vocabulary has exactly one synset, so no disambiguation is required
for those words. For the polysemous tail, frequency evidence only
exists for a minority of words — any future f that attempts sense
selection must either rely on that minority's lemma counts or on a
signal other than WordNet's built-in ordering.

**Outputs.**
- `outputs/pos_wordnet_census-results.md`
- `outputs/pos_mismatch_examples.md`
- `outputs/figures/pos_vocab_distribution.png`
- `outputs/figures/pos_triplet_composition.png`
- `outputs/figures/pos_validation_composition.png`

## §8 — Follow-up: sense[0] POS reliability by POS, for the wndef and wnex vocabularies

An exploratory view of how well `synsets(word)[0]`'s POS represents the
word's most common meaning, broken down by the sense[0] POS itself.
Five ordered reliability categories (most to least confident) are
computed per word from WordNet lemma counts:

1. **unique POS** — all synsets share sense[0]'s POS; there is no POS
   ambiguity.
2. **most common POS** — multiple POS are present, and the POS of
   sense[0] holds the maximum lemma count (held either by sense[0]
   itself or by another synset with the same POS).
3. **POS tie** — sense[0]'s POS shares the top lemma count with at
   least one synset in a different POS; the max count is > 0.
4. **POS tie — no lemmas** — every synset has zero lemma count, so no
   POS is distinguishable by frequency.
5. **NOT most common POS** — some other POS has a strictly higher
   lemma count than anything in sense[0]'s POS.

a+s (adjective and satellite adjective) are combined as `adj` for this
analysis. The plot is produced for `vocabulary_wndef.csv` (the full
wndef-scope vocabulary) and `vocabulary_wnex.csv` (the full wnex-scope
vocabulary) so that the POS reliability profile of each phrase
construction strategy's vocabulary is visible side by side.

In [ ]:
# === §8: Stacked-bar POS reliability for wndef and wnex vocabularies
RELIABILITY_CATEGORIES = [
    'unique POS',
    'most common POS',
    'POS tie',
    'POS tie — no lemmas',
    'NOT most common POS',
]
POS_ORDER_OUT  = ['n', 'v', 'a', 'r']
POS_LABELS_OUT = {'n': 'noun', 'v': 'verb', 'a': 'adj', 'r': 'adv'}

def classify_pos_reliability(word):
    """Return (sense0_pos_combined, reliability_category).
    Classification is at the POS level: per-POS max lemma count across
    a word's synsets (a+s combined as 'a'), compared to sense[0]'s POS.
    """
    syns = wn.synsets(word)
    if not syns:
        return None, None
    pos_max = {}
    for s in syns:
        p = _combine_adj(s.pos())
        c = sum(l.count() for l in s.lemmas() if l.name() == word)
        if c > pos_max.get(p, -1):
            pos_max[p] = c
    p0 = _combine_adj(syns[0].pos())
    if len(pos_max) == 1:
        return p0, 'unique POS'
    overall_max = max(pos_max.values())
    if overall_max == 0:
        return p0, 'POS tie — no lemmas'
    winners = {p for p, v in pos_max.items() if v == overall_max}
    if p0 not in winners:
        return p0, 'NOT most common POS'
    if len(winners) > 1:
        return p0, 'POS tie'
    return p0, 'most common POS'

def build_reliability_df(vocab_df):
    recs = [classify_pos_reliability(w) for w in vocab_df['word']]
    return pd.DataFrame(recs, columns=['sense0_pos_combined', 'reliability'])

vocab_wndef = pd.read_csv(
    DATA_DIR / 'filtered_split' / 'wn_synset' / 'wndef' / 'vocabulary_wndef.csv',
    keep_default_na=False, na_values=[''],
)
vocab_wnex = pd.read_csv(
    DATA_DIR / 'filtered_split' / 'wn_synset' / 'wnex' / 'vocabulary_wnex.csv',
    keep_default_na=False, na_values=[''],
)
print(f'vocabulary_wndef: {len(vocab_wndef):,} words')
print(f'vocabulary_wnex:  {len(vocab_wnex):,} words')

t0 = time.time()
rel_wndef = build_reliability_df(vocab_wndef)
rel_wnex  = build_reliability_df(vocab_wnex)
print(f'Classified {len(rel_wndef) + len(rel_wnex):,} words in '
      f'{time.time() - t0:.1f}s')

def stacked_reliability_bar(ax, rel_df, title):
    ct = pd.crosstab(
        rel_df['sense0_pos_combined'],
        rel_df['reliability'],
    ).reindex(index=POS_ORDER_OUT, columns=RELIABILITY_CATEGORIES, fill_value=0)
    # Green → amber → grey → red, ordered most- to least-confident.
    colors = ['#1a9850', '#a6d96a', '#fee08b', '#bababa', '#d73027']
    x = np.arange(len(POS_ORDER_OUT))
    bottom = np.zeros(len(POS_ORDER_OUT))
    for cat, color in zip(RELIABILITY_CATEGORIES, colors):
        vals = ct[cat].values
        ax.bar(x, vals, bottom=bottom, label=cat, color=color, edgecolor='black')
        bottom += vals
    # Total-count annotations above each bar
    totals = ct.sum(axis=1).values
    for xi, tot in zip(x, totals):
        ax.text(xi, tot + max(totals) * 0.01, f'{int(tot):,}', ha='center', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([POS_LABELS_OUT[p] for p in POS_ORDER_OUT])
    ax.set_xlabel('sense[0] POS')
    ax.set_ylabel('Number of words')
    ax.set_title(f'{title}  (n={len(rel_df):,})')
    ax.set_ylim(0, max(totals) * 1.1)
    return ct

FIG_POS_REL = FIGURE_DIR / 'pos_reliability_stacked.png'
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ct_wndef = stacked_reliability_bar(axes[0], rel_wndef, 'vocabulary_wndef')
ct_wnex  = stacked_reliability_bar(axes[1], rel_wnex,  'vocabulary_wnex')
axes[1].legend(title='Reliability', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(FIG_POS_REL, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_POS_REL}')

print()
print('wndef crosstab (rows = sense[0] POS, cols = reliability):')
print(ct_wndef)
print()
print('wnex crosstab:')
print(ct_wnex)

## §9 — Follow-up: sense[0] reliability at the synset level

A synset-level companion to §8. Rather than asking whether sense[0]'s
POS is the most common POS for the word, here we ask whether sense[0]
is the most common *synset* for the word — a strictly tighter check,
since two synsets sharing sense[0]'s POS are still distinct meanings
and picking the wrong one is still a sense-selection error.

Five ordered categories (most to least confident):

1. **Unique Sense** — sense[0] is the only synset.
2. **Most Common** — multi-synset and sense[0] has a strictly higher
   lemma count than any other synset.
3. **Sense Tie** — sense[0] is tied with at least one other synset for
   the highest lemma count, and has at least one lemma.
4. **Tie — No Lemmas** — multi-synset and every synset has zero lemma
   count (no evidence to rank them).
5. **NOT Most Common** — some other synset has a strictly higher lemma
   count than sense[0].

One bar chart per vocabulary (`vocabulary_wndef.csv` and
`vocabulary_wnex.csv`), side by side.

In [ ]:
# === §9: Synset-level sense[0] reliability for wndef and wnex vocabularies
SENSE_CATEGORIES = [
    'Unique Sense',
    'Most Common',
    'Sense Tie',
    'Tie — No Lemmas',
    'NOT Most Common',
]

def classify_sense_reliability(word):
    syns = wn.synsets(word)
    if not syns:
        return None
    if len(syns) == 1:
        return 'Unique Sense'
    counts = [sum(l.count() for l in s.lemmas() if l.name() == word) for s in syns]
    s0_count = counts[0]
    other_max = max(counts[1:])
    if s0_count > other_max:
        return 'Most Common'
    if s0_count < other_max:
        return 'NOT Most Common'
    # tied with another synset
    if s0_count == 0:
        return 'Tie — No Lemmas'
    return 'Sense Tie'

t0 = time.time()
sense_wndef = vocab_wndef['word'].map(classify_sense_reliability)
sense_wnex  = vocab_wnex['word'].map(classify_sense_reliability)
print(f'Classified {len(sense_wndef) + len(sense_wnex):,} words in '
      f'{time.time() - t0:.1f}s')

def sense_bars(ax, series, title):
    counts = series.value_counts().reindex(SENSE_CATEGORIES, fill_value=0)
    colors = ['#1a9850', '#a6d96a', '#fee08b', '#bababa', '#d73027']
    bars = ax.bar(range(len(SENSE_CATEGORIES)), counts.values,
                  color=colors, edgecolor='black')
    ax.set_xticks(range(len(SENSE_CATEGORIES)))
    ax.set_xticklabels(SENSE_CATEGORIES, rotation=25, ha='right')
    ax.set_ylabel('Number of words')
    ax.set_title(f'{title}  (n={len(series):,})')
    top = counts.max()
    for i, v in enumerate(counts.values):
        ax.text(i, v + top * 0.01,
                f'{v:,}\n({v/len(series):.1%})',
                ha='center', fontsize=9)
    ax.set_ylim(0, top * 1.15)
    return counts

FIG_SENSE_REL = FIGURE_DIR / 'sense_reliability_bars.png'
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
counts_wndef = sense_bars(axes[0], sense_wndef, 'vocabulary_wndef')
counts_wnex  = sense_bars(axes[1], sense_wnex,  'vocabulary_wnex')
fig.tight_layout()
fig.savefig(FIG_SENSE_REL, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_SENSE_REL}')

print()
print('wndef category counts:')
print(counts_wndef)
print()
print('wnex category counts:')
print(counts_wnex)